# gold_payments_by_status

**Source:** `silver.payments`  
**Grain:** one row per payment status value  
**Merge key:** `status`  
**Lineage:** Silver → Gold (never Bronze directly)

In [ ]:
dbutils.widgets.removeAll()
dbutils.widgets.text("catalog", "ubereats_dev")

In [ ]:
catalog      = dbutils.widgets.get("catalog")
gold_table   = f"{catalog}.gold.payments_by_status"
silver_src   = f"{catalog}.silver.payments"

print(f"[gold] {gold_table}  ←  {silver_src}")

In [ ]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {gold_table} (
        status                  STRING NOT NULL,
        payment_count           LONG,
        total_amount_brl        DOUBLE,
        avg_amount_brl          DOUBLE,
        total_net_amount_brl    DOUBLE,
        total_platform_fee_brl  DOUBLE,
        total_refund_amount_brl DOUBLE,
        total_tax_amount_brl    DOUBLE,
        _computed_at            TIMESTAMP NOT NULL
    ) USING DELTA
    CLUSTER BY (status)
    TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')
""")
print(f"[gold] table {gold_table} ready")

In [ ]:
from pyspark.sql.functions import count, sum, avg, current_timestamp

# fact: silver.payments — aggregation by status
agg_df = (
    spark.table(silver_src)
    .groupBy("status")
    .agg(
        count("payment_id").alias("payment_count"),
        sum("amount").alias("total_amount_brl"),
        avg("amount").alias("avg_amount_brl"),
        sum("net_amount").alias("total_net_amount_brl"),
        sum("platform_fee").alias("total_platform_fee_brl"),
        sum("refund_amount").alias("total_refund_amount_brl"),
        sum("tax_amount").alias("total_tax_amount_brl"),
    )
    .withColumn("_computed_at", current_timestamp())
)

print(f"[gold] aggregated {agg_df.count()} status values")

In [ ]:
agg_df.createOrReplaceTempView("gold_payments_by_status_batch")

spark.sql(f"""
    MERGE INTO {gold_table} AS t
    USING gold_payments_by_status_batch AS s
    ON t.status = s.status
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""")

print(f"[gold] {gold_table} updated")